# Brussels Post-Processing Pipeline

Applies post-detection filters to M-DRAC conflicts:
1. Teleportation filter - removes conflicts during tracking glitches
2. Pair deduplication - keeps worst-case detection per pair

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

# Modular imports
from ssm.utils import load_config
from filters.postprocessing.teleportation_filter import filter_conflicts_by_teleportation
from utils.io_helpers import save_detection_results

print("✓ Imports complete")

## Configuration

In [ ]:
# Load configuration
config = load_config("/home/ubuntu/prem/config.yaml")

# Paths
REGION = "brussels"
START_DATE = "2025-06-04"
DATA_DIR = "/home/ubuntu/data/uploads/objects/clean"
RESULTS_DIR = "/home/ubuntu/prem/results/brussels"

# Input files (from main pipeline)
MDRAC_INPUT = f"{RESULTS_DIR}/{REGION}/mdrac/04/mdrac_04.csv"

# Output
OUTPUT_DIR = f"{RESULTS_DIR}/{REGION}"

print(f"Configuration:")
print(f"  Region: {REGION}")
print(f"  Date: {START_DATE}")
print(f"  M-DRAC input: {MDRAC_INPUT}")
print(f"  Output: {OUTPUT_DIR}")

## Utilities

In [ ]:
def create_pair_id(df: pd.DataFrame) -> pd.DataFrame:
    """
    Create normalized pair_id (always smaller ID first).
    Ensures (id1=100, id2=200) and (id1=200, id2=100) are treated as same pair.
    """
    df = df.copy()
    df['pair_id'] = df.apply(
        lambda r: f"{min(r['id1'], r['id2'])}_{max(r['id1'], r['id2'])}",
        axis=1
    )
    return df


def deduplicate_pairs(df: pd.DataFrame, metric_col: str = 'MDRAC') -> pd.DataFrame:
    """
    Group by pair_id and select worst case.
    
    Args:
        df: DataFrame with pair_id column
        metric_col: Column to maximize ('MDRAC' or 'composite_risk')
    
    Returns:
        Deduplicated DataFrame (one row per pair)
    """
    initial_count = len(df)
    unique_pairs = df['pair_id'].nunique()
    
    # Keep worst-case detection per pair
    deduplicated = df.loc[df.groupby('pair_id')[metric_col].idxmax()].copy()
    
    reduction = 100 * (initial_count - len(deduplicated)) / initial_count
    print(f"  Deduplication: {initial_count:,} → {len(deduplicated):,} ({reduction:.1f}% reduction)")
    print(f"  Avg detections per pair: {initial_count/unique_pairs:.1f}")
    
    return deduplicated

---
# M-DRAC Post-Processing

## Load M-DRAC Detections

In [ ]:
print("="*70)
print("M-DRAC POST-PROCESSING")
print("="*70)

mdrac_raw = pd.read_csv(MDRAC_INPUT)
print(f"\nLoaded {len(mdrac_raw)} raw M-DRAC detections")
print(f"Columns: {list(mdrac_raw.columns)}")
print(f"\nSample:")
mdrac_raw.head(3)

## 1. Teleportation Filter

In [ ]:
print("\n" + "="*70)
print("TELEPORTATION FILTER")
print("="*70)

# Load vehicle data for the same date
vehicle_data_path = f"{DATA_DIR}/{START_DATE}/objects.parquet"
print(f"Loading vehicle data: {vehicle_data_path}")

try:
    df = pd.read_parquet(vehicle_data_path)
    print(f"✓ Loaded {len(df):,} vehicle records")
    
    # Apply teleportation filter
    mdrac_clean = filter_conflicts_by_teleportation(
        conflicts=mdrac_raw,
        vehicle_data=df,
        max_jump=config['postprocessing']['teleportation_filter'].get('max_speed_threshold', 50.0),
        verbose=True
    )
    
    removed = len(mdrac_raw) - len(mdrac_clean)
    print(f"\n✓ Removed {removed} conflicts with teleportation glitches")
    
except FileNotFoundError:
    print(f"⚠️  Vehicle data not found: {vehicle_data_path}")
    print("  Skipping teleportation filter")
    mdrac_clean = mdrac_raw.copy()

## 2. Pair Deduplication

In [ ]:
print("\n" + "="*70)
print("PAIR DEDUPLICATION")
print("="*70)

# Create pair IDs
mdrac_clean = create_pair_id(mdrac_clean)

# Deduplicate (keep worst case per pair)
mdrac_final = deduplicate_pairs(mdrac_clean, metric_col='MDRAC')

print(f"\n✓ Final M-DRAC conflicts: {len(mdrac_final):,}")

## 3. Save Results

In [ ]:
print("\n" + "="*70)
print("SAVING RESULTS")
print("="*70)

# Save post-processed results
output_path = save_detection_results(
    mdrac_final,
    OUTPUT_DIR,
    'mdrac_postprocessed',
    REGION,
    START_DATE
)

print(f"\n✓ Saved {len(mdrac_final)} post-processed conflicts")
print(f"  Output: {output_path}")

# Summary
print("\n" + "="*70)
print("SUMMARY")
print("="*70)
print(f"  Raw detections: {len(mdrac_raw):,}")
print(f"  After teleportation filter: {len(mdrac_clean):,}")
print(f"  After deduplication: {len(mdrac_final):,}")
print(f"  Total removed: {len(mdrac_raw) - len(mdrac_final):,} ({100*(len(mdrac_raw)-len(mdrac_final))/len(mdrac_raw):.1f}%)")
print("="*70)

## Analysis

In [ ]:
# Zone distribution
print("\nZone Distribution:")
print(mdrac_final['zone'].value_counts())

# Severity distribution  
print("\nSeverity Distribution:")
severity_bins = [0, 5.0, 7.0, float('inf')]
severity_labels = ['moderate', 'severe', 'critical']
mdrac_final['severity_bin'] = pd.cut(mdrac_final['MDRAC'], bins=severity_bins, labels=severity_labels)
print(mdrac_final['severity_bin'].value_counts())

# Top 10 conflicts
print("\nTop 10 Critical Conflicts:")
mdrac_final.nlargest(10, 'MDRAC')[['timestamp', 'zone', 'interaction', 'MDRAC', 'TTC', 'dist']]